# 06 · Opt-in fresh GPU experiment and its measured reference
**Default execution is offline.** This notebook displays the exact full-cell command and
its already measured reference rows. Change `RUN_GPU = True` only after following
`docs/reproduction.md`, downloading the exact data/model, and installing the GPU extras.
One full Bolt-S / ETTh1 / H96 / seed-0 cell trains lightweight modules and runs all methods.
It can take significant GPU time; the notebook does not automatically start all 432 cells.


In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'artifacts/manifest.json').exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ttc4ts.results import (checked_path, load_snapshot, geometric, summarize,
    main_summary, lw_budget_summary, uncertainty_summary, MODELS, MODEL_LABELS, METHODS, METHOD_LABELS)
%matplotlib inline
plt.rcParams.update({'figure.dpi': 115, 'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .18})
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', lambda x: f'{x:.5f}')
print('Offline mode: numerical snapshots are verified against the released SHA-256 manifest.')


Offline mode: numerical snapshots are verified against the released SHA-256 manifest.


In [2]:
RUN_GPU = False
MODEL, DATASET, HORIZON, SEED = 'chronos-bolt-small', 'ETTh1', 96, 0
reference = load_snapshot('main')
reference = reference[(reference.model==MODEL) & (reference.dataset==DATASET) &
    (reference.horizon==HORIZON) & (reference.seed==SEED) & reference.method.isin(METHODS)]
display(reference[['method','mse','mae','relative_mse','relative_mae']])
command = [sys.executable, 'scripts/run_experiment.py', '--model', MODEL, '--dataset', DATASET,
    '--horizon', str(HORIZON), '--seed', str(SEED), '--output', 'results_notebook_reproduction',
    '--adapter-steps', '32', '--aggregator-steps', '160', '--budget-aggregator-steps', '40']
# Print a portable command, never an interpreter's private machine path.
print('python ' + ' '.join(command[1:]))


,method,mse,mae,relative_mse,relative_mae
29913,backbone,0.89236,0.48650,1.00000,1.00000
29914,uniform,0.86695,0.48345,0.97152,0.99373
29997,validation_best,0.89228,0.49867,0.99990,1.02500
29998,weighted,0.81340,0.48070,0.91152,0.98807
29999,classifier,1.14821,0.57220,1.28671,1.17614
30000,risk,0.87655,0.48415,0.98228,0.99517
30001,ranking,0.84505,0.50680,0.94699,1.04172
30002,greedy,0.81347,0.48543,0.91159,0.99779
30003,retrospective,0.88558,0.48481,0.99240,0.99651
30004,validation_selected,0.81347,0.48543,0.91159,0.99779


python scripts/run_experiment.py --model chronos-bolt-small --dataset ETTh1 --horizon 96 --seed 0 --output results_notebook_reproduction --adapter-steps 32 --aggregator-steps 160 --budget-aggregator-steps 40


In [3]:
if RUN_GPU:
    import subprocess, torch
    assert torch.cuda.is_available(), 'Install the GPU environment first.'
    subprocess.run([sys.executable, 'scripts/check_data.py', '--study', 'main'], cwd=ROOT, check=True)
    cell = f'{MODEL}_{DATASET}_h{HORIZON}_s{SEED}'
    destination = ROOT/'results_notebook_reproduction'/cell
    if destination.exists():
        raise FileExistsError('Choose a new output directory or inspect the existing cell before repeating.')
    subprocess.run(command, cwd=ROOT, check=True)
    metrics = json.loads((destination/'metrics.json').read_text())
    print('Fresh cell completed. Inspect its metrics and source hashes before comparison.')
    display(pd.DataFrame(metrics['results']).T)
else:
    print('GPU branch disabled. The table above is the checked historical measurement, not a new run.')


GPU branch disabled. The table above is the checked historical measurement, not a new run.


## Full study commands
See `docs/reproduction.md` for the 432-cell campaign, 72 Adapter-grid cells, 54 fixed
Learned-weights budget cells, independent baselines and 108 UQ cells. Audits must pass before
labeling any campaign complete. Some original Hub revisions are not retained, so a newly
downloaded model can differ even with the same repository name. Compare the recorded
frozen-state digest, package versions, dataset hashes, candidate names and split origins.

Offline notebook execution is not evidence that this GPU branch has been run on a new machine.
